# Alerts vs Alerts Backup – Discrepancy Report

Compares **source (alerts)** and **backup** tables by `ingestion_date` (and `ingestion_timestamp`) and reports:
- Count mismatches: distinct `customersourceuniqueid` per (ingestion_date, ingestion_timestamp)
- IDs only in source vs only in backup
- **Latest vs previous run:** per ingestion_date, when the two most recent ingestion_timestamps have different distinct customer counts

Results are saved to a JSON file for use before/after running deduplication (backup with `updateTables=True`).

**Configuration:** Schema, catalog, table names, and MinIO bucket are read from **environment variables** (or set in the CONFIG cell). See section 1. This avoids hardcoded names.

## 1. Setup and configuration

In [ ]:
from pipeline_utils.utils import get_env_creds
from pipeline_utils.logging_utils import get_logger
import json
from datetime import datetime
import pandas as pd

def ts_to_readable(ts):
    """Convert Unix ingestion_timestamp to human-readable UTC string for debugging."""
    if ts is None or (isinstance(ts, float) and pd.isna(ts)):
        return ""
    try:
        return datetime.utcfromtimestamp(int(ts)).strftime("%Y-%m-%d %H:%M:%S UTC")
    except (ValueError, OSError):
        return str(ts)

In [ ]:
import os

# --- CONFIG: use environment variables to avoid hardcoded names (recommended for shared/tax-documentation repo). ---
# Set env vars or assign below for local run; required for execution.
SCHEMA = os.environ.get("ALERTS_REPORT_SCHEMA")  # e.g. your eureka schema; set ALERTS_REPORT_SCHEMA or assign here
CATALOG = os.environ.get("ALERTS_REPORT_CATALOG")  # e.g. catalog name; set ALERTS_REPORT_CATALOG or assign here
SOURCE_TABLE = os.environ.get("ALERTS_REPORT_SOURCE_TABLE") or "alerts"  # default generic table name
ID_COLUMN = None # defined by user
DATE_COL = None # defined by user
TS_COL = None # defined by user

# Backup table: set via env ALERTS_REPORT_BACKUP_TABLE or leave None to auto-detect latest source_backup_*
BACKUP_TABLE = os.environ.get("ALERTS_REPORT_BACKUP_TABLE")  # None = auto-detect

# Output paths
OUTPUT_JSON = os.environ.get("ALERTS_REPORT_OUTPUT_JSON") or "alerts_backup_discrepancies.json"
ALERTS_DIFF_CATALOG = os.environ.get("ALERTS_REPORT_DIFF_CATALOG") or "alerts_diff"  # folder for per-day CSV files

# MinIO bucket for "MinIO: ALERTS files for mismatch dates" section; set to skip that section
MINIO_BUCKET_ALERTS = os.environ.get("ALERTS_REPORT_MINIO_BUCKET")  # None = skip MinIO listing

# Optional: cap listed IDs per discrepancy to avoid huge JSON (None = no cap)
MAX_IDS_PER_DISCREPANCY = 1000

if SCHEMA is None:
    raise ValueError("SCHEMA is not set. Set env ALERTS_REPORT_SCHEMA or assign SCHEMA in this cell.")
if CATALOG is None:
    raise ValueError("CATALOG is not set. Set env ALERTS_REPORT_CATALOG or assign CATALOG in this cell.")

In [ ]:
# Get Spark session (assumes already set up in environment, e.g. from stack)
from sdk import create_spark_session, delete_spark_session
delete_spark_session()
spark = create_spark_session(
    enable_spot_instance=True,
    spark_params={
        "executor.cores": 6,
        "executor.instances": 12,
        "executor.memory": "32g",
        "spark.sql.autoBroadcastJoinThreshold": -1,
    },
)
logger = get_logger()

# catalog and full_schema from CONFIG (env or assigned above)
full_schema = f"{CATALOG}.{SCHEMA}"

In [ ]:
# Resolve backup table name if not set
if BACKUP_TABLE is None:
    tables_df = spark.sql(f"SHOW TABLES IN {full_schema}").toPandas()
    tables = tables_df["tableName"].astype(str).tolist()
    backup_candidates = [t for t in tables if t.startswith(SOURCE_TABLE + "_backup_")]
    if not backup_candidates:
        raise ValueError(f"No backup table found for {SOURCE_TABLE} in {full_schema}. Set BACKUP_TABLE manually.")
    # Use latest by name (backup name usually includes timestamp)
    BACKUP_TABLE = sorted(backup_candidates)[-1]
    print(f"Using backup table: {BACKUP_TABLE}")
else:
    print(f"Using backup table: {BACKUP_TABLE}")

**MinIO ALERTS listing** is in the section **"MinIO: ALERTS files for mismatch dates"** below. Run **section 2c** first so that ingestion_dates with timestamp mismatches are available.

In [ ]:
# Run section 2c first. Then run the "MinIO: ALERTS files for mismatch dates" section below.

## 2. Counts per ingestion_date and ingestion_timestamp

In [ ]:
def get_counts_by_date_and_ts(spark, full_schema, table, date_col, ts_col, id_col):
    q = f"""
    SELECT `{date_col}`, `{ts_col}`,
           count(DISTINCT `{id_col}`) AS distinct_count
    FROM {full_schema}.{table}
    WHERE `{id_col}` IS NOT NULL
    GROUP BY `{date_col}`, `{ts_col}`
    ORDER BY `{date_col}`, `{ts_col}`
    """
    return spark.sql(q).toPandas()

source_counts = get_counts_by_date_and_ts(spark, full_schema, SOURCE_TABLE, DATE_COL, TS_COL, ID_COLUMN)
source_counts = source_counts.rename(columns={"distinct_count": "source_count"})
source_counts["ingestion_timestamp_readable"] = source_counts[TS_COL].apply(ts_to_readable)

backup_counts = get_counts_by_date_and_ts(spark, full_schema, BACKUP_TABLE, DATE_COL, TS_COL, ID_COLUMN)
backup_counts = backup_counts.rename(columns={"distinct_count": "backup_count"})
backup_counts["ingestion_timestamp_readable"] = backup_counts[TS_COL].apply(ts_to_readable)

print("Source (alerts) counts by ingestion_date, ingestion_timestamp:")
display(source_counts)
print("Backup counts by ingestion_date, ingestion_timestamp:")
display(backup_counts)

### 2b. Counts per ingestion_date only (distinct IDs per day)

In [ ]:
def get_counts_by_date_only(spark, full_schema, table, date_col, id_col):
    q = f"""
    SELECT `{date_col}`,
           count(DISTINCT `{id_col}`) AS distinct_count
    FROM {full_schema}.{table}
    WHERE `{id_col}` IS NOT NULL
    GROUP BY `{date_col}`
    ORDER BY `{date_col}`
    """
    return spark.sql(q).toPandas()

source_counts_by_date = get_counts_by_date_only(spark, full_schema, SOURCE_TABLE, DATE_COL, ID_COLUMN)
source_counts_by_date = source_counts_by_date.rename(columns={"distinct_count": "source_count"})
backup_counts_by_date = get_counts_by_date_only(spark, full_schema, BACKUP_TABLE, DATE_COL, ID_COLUMN)
backup_counts_by_date = backup_counts_by_date.rename(columns={"distinct_count": "backup_count"})

by_date = source_counts_by_date.merge(backup_counts_by_date, on=DATE_COL, how="outer", indicator=True)
by_date["source_count"] = by_date["source_count"].fillna(0).astype(int)
by_date["backup_count"] = by_date["backup_count"].fillna(0).astype(int)
by_date["diff"] = by_date["source_count"] - by_date["backup_count"]
by_date["status"] = by_date["diff"].apply(lambda x: "match" if x == 0 else "mismatch")
by_date = by_date.drop(columns=["_merge"])

print("Per ingestion_date (distinct customersourceuniqueid):")
display(by_date)
date_discrepancies = by_date[by_date["status"] == "mismatch"]
print(f"Dates with count mismatch: {len(date_discrepancies)}")

### 2c. Latest vs each other ingestion_timestamp (per ingestion_date)

For each ingestion_date, compare the **most recent** run to **every other** run (each older ingestion_timestamp separately). When the number of distinct `customersourceuniqueid` differs between latest and that run, record a discrepancy. This lists all (date, other_timestamp) pairs with different coverage.

In [ ]:
# Per ingestion_date: compare latest timestamp to each other timestamp separately (source table)
latest_vs_previous_list = []
for ingestion_date, grp in source_counts.groupby(DATE_COL, sort=False):
    grp = grp.sort_values(TS_COL, ascending=False).reset_index(drop=True)
    if len(grp) < 2:
        continue
    ts_most = int(grp.iloc[0][TS_COL])
    count_most = int(grp.iloc[0]["source_count"])
    for i in range(1, len(grp)):
        other = grp.iloc[i]
        ts_other = int(other[TS_COL])
        count_other = int(other["source_count"])
        if count_most != count_other:
            latest_vs_previous_list.append({
                DATE_COL: str(ingestion_date),
                "most_recent_ingestion_timestamp": ts_most,
                "previous_ingestion_timestamp": ts_other,
                "most_recent_run_at": ts_to_readable(ts_most),
                "previous_run_at": ts_to_readable(ts_other),
                "count_most_recent": count_most,
                "count_previous": count_other,
                "diff": count_most - count_other,
            })

latest_vs_previous_df = pd.DataFrame(latest_vs_previous_list)
if len(latest_vs_previous_df) > 0:
    print("Latest vs each other run (per ingestion_date) where distinct customer counts differ:")
    display(latest_vs_previous_df)
    print(f"Total: {len(latest_vs_previous_df)} (date, other_timestamp) pairs with count mismatch.")
else:
    print("No (ingestion_date, other run) pairs with different distinct customer counts.")

In [ ]:
# Display actual rows that differ between latest and previous run (IDs only in one run)
# and save each day's case to a separate CSV in alerts_diff/
import os

def get_distinct_ids_for_ts(spark, full_schema, table, date_col, ts_col, id_col, date_val, ts_val):
    q = f"""
    SELECT DISTINCT `{id_col}` AS id_val
    FROM {full_schema}.{table}
    WHERE `{date_col}` = '{date_val}' AND `{ts_col}` = {int(ts_val)} AND `{id_col}` IS NOT NULL
    """
    return set(r["id_val"] for r in spark.sql(q).collect())

# ALERTS_DIFF_CATALOG from CONFIG (env ALERTS_REPORT_DIFF_CATALOG or default)
MAX_IDS_DISPLAY = 300  # cap for SQL IN clause when fetching rows (increase to get more rows in CSV)

os.makedirs(ALERTS_DIFF_CATALOG, exist_ok=True)

all_diffs = []  # collect all df_diff to save into one file

if len(latest_vs_previous_list) > 0:
    for rec in latest_vs_previous_list:
        d = rec[DATE_COL]
        ts_most = rec["most_recent_ingestion_timestamp"]
        ts_prev = rec["previous_ingestion_timestamp"]
        ids_most = get_distinct_ids_for_ts(spark, full_schema, SOURCE_TABLE, DATE_COL, TS_COL, ID_COLUMN, d, ts_most)
        ids_prev = get_distinct_ids_for_ts(spark, full_schema, SOURCE_TABLE, DATE_COL, TS_COL, ID_COLUMN, d, ts_prev)
        only_in_latest = ids_most - ids_prev
        only_in_previous = ids_prev - ids_most
        if not only_in_latest and not only_in_previous:
            continue
        ids_to_show = list(only_in_latest | only_in_previous)[:MAX_IDS_DISPLAY]
        in_clause = ", ".join(repr(str(x)) for x in ids_to_show)
        q_rows = f"""
        SELECT *, CASE WHEN `{TS_COL}` = {ts_most} THEN 'latest' ELSE 'previous' END AS run_type
        FROM {full_schema}.{SOURCE_TABLE}
        WHERE `{DATE_COL}` = '{d}' AND `{TS_COL}` IN ({ts_most}, {ts_prev})
          AND `{ID_COLUMN}` IN ({in_clause})
        ORDER BY run_type, `{ID_COLUMN}`
        """
        df_diff = spark.sql(q_rows).toPandas()
        if TS_COL in df_diff.columns:
            df_diff["ingestion_timestamp_readable"] = df_diff[TS_COL].apply(ts_to_readable)
        n_total = len(only_in_latest) + len(only_in_previous)
        cap_note = f" (showing up to {MAX_IDS_DISPLAY} IDs)" if n_total > MAX_IDS_DISPLAY else ""
        print(f"--- {DATE_COL}={d} | latest: {ts_to_readable(ts_most)} | previous: {ts_to_readable(ts_prev)} | {len(only_in_latest)} only in latest, {len(only_in_previous)} only in previous{cap_note} ---")
        display(df_diff)
        # Save this comparison to a separate CSV (one per date and other timestamp)
        csv_path = os.path.join(ALERTS_DIFF_CATALOG, f"{d}_{ts_prev}.csv")
        df_diff.to_csv(csv_path, index=False)
        print(f"Saved to {csv_path} ({len(df_diff)} rows)")
        all_diffs.append(df_diff)

    # Save all differing rows to one combined file
    if all_diffs:
        combined_df = pd.concat(all_diffs, ignore_index=True)
        combined_path = os.path.join(ALERTS_DIFF_CATALOG, "alerts_diff_all.csv")
        combined_df.to_csv(combined_path, index=False)
        print(f"Saved all differing rows to {combined_path} ({len(combined_df)} rows)")
else:
    print("No latest-vs-other discrepancies; nothing to display.")

## MinIO: ALERTS files for mismatch dates (from 2c)

For **ingestion_date** values that have latest-vs-previous timestamp mismatches (section 2c), list the corresponding ALERTS objects in the configured MinIO bucket (set `ALERTS_REPORT_MINIO_BUCKET` env or `MINIO_BUCKET_ALERTS` in CONFIG) with object name and last modification time. Use this to check when the MinIO file that stores alerts was last modified for each problematic date. If the bucket is not set, this section is skipped.

In [ ]:
# Ingestion dates from section 2c (latest-vs-previous mismatches only)
# MINIO_BUCKET_ALERTS from CONFIG (env ALERTS_REPORT_MINIO_BUCKET); None = skip this section
problem_ingestion_dates = [rec[DATE_COL] for rec in latest_vs_previous_list]

if len(problem_ingestion_dates) == 0:
    print("No mismatch dates from section 2c; nothing to list in MinIO.")
    minio_alerts_df = pd.DataFrame(columns=[DATE_COL, "object_name", "last_modified"])
elif MINIO_BUCKET_ALERTS is None:
    print("MINIO_BUCKET_ALERTS not set (env ALERTS_REPORT_MINIO_BUCKET). Skipping MinIO listing.")
    minio_alerts_df = pd.DataFrame(columns=[DATE_COL, "object_name", "last_modified"])
else:
    import re
    from pipeline_utils.storage_api import StorageConnector

    problem_dates_set = set(str(d) for d in problem_ingestion_dates)

    def parse_ingestion_date_from_name(name):
        match = re.search(r"(\d{8})", name)
        return match.group(1) if match else ""

    env_creds = get_env_creds(SCHEMA)
    storage_config = env_creds["storage_config"]
    sc = StorageConnector(storage_config)
    client = sc.connector.client

    objects = client.list_objects(MINIO_BUCKET_ALERTS, recursive=True)
    alerts_objects = [obj for obj in objects if "ALERTS" in obj.object_name]

    rows = []
    for obj in alerts_objects:
        parsed_date = parse_ingestion_date_from_name(obj.object_name)
        if parsed_date in problem_dates_set:
            rows.append({
                DATE_COL: parsed_date,
                "object_name": obj.object_name,
                "last_modified": obj.last_modified,
            })

    minio_alerts_df = pd.DataFrame(rows)
    if len(minio_alerts_df) > 0:
        minio_alerts_df = minio_alerts_df.sort_values([DATE_COL, "last_modified"], ascending=[True, False]).reset_index(drop=True)
        minio_alerts_df["last_modified"] = minio_alerts_df["last_modified"].astype(str)

    print(f"ALERTS objects in bucket '{MINIO_BUCKET_ALERTS}' for {len(problem_dates_set)} mismatch ingestion_dates (from 2c): {len(minio_alerts_df)} objects")
display(minio_alerts_df)

## 3. Join and find count discrepancies

In [ ]:
merge_cols = [DATE_COL, TS_COL]
combined = source_counts.merge(
    backup_counts,
    on=merge_cols,
    how="outer",
    indicator=True
)
combined["source_count"] = combined["source_count"].fillna(0).astype(int)
combined["backup_count"] = combined["backup_count"].fillna(0).astype(int)
combined["diff"] = combined["source_count"] - combined["backup_count"]
combined["status"] = combined["diff"].apply(lambda x: "match" if x == 0 else "mismatch")
combined["ingestion_timestamp_readable"] = combined[TS_COL].apply(ts_to_readable)

# Rows only in source or only in backup
combined["in_source_only"] = combined["_merge"] == "left_only"
combined["in_backup_only"] = combined["_merge"] == "right_only"
combined = combined.drop(columns=["_merge"])

discrepancies_df = combined[combined["status"] == "mismatch"]
print(f"Total (date, timestamp) groups: {len(combined)}")
print(f"Discrepancies (count mismatch or only in one side): {len(discrepancies_df)}")
display(combined)

### 3b. Source vs backup: distinct count by ingestion_date, ingestion_timestamp and source_system

Compare **source (alerts)** and **backup** tables by distinct count of `customersourceuniqueid` broken down by **SOURCE_SYSTEM**. For each (ingestion_date, ingestion_timestamp, source_system), shows source_count, backup_count, diff and match/mismatch status.

In [ ]:
SOURCE_SYSTEM_COL = "SOURCE_SYSTEM"

def get_counts_by_date_ts_source_system(spark, full_schema, table, date_col, ts_col, source_system_col, id_col):
    q = f"""
    SELECT `{date_col}`, `{source_system_col}`,
           count(DISTINCT `{id_col}`) AS distinct_count
    FROM {full_schema}.{table}
    WHERE `{id_col}` IS NOT NULL
    GROUP BY `{date_col}`, `{source_system_col}`
    ORDER BY `{date_col}`, `{source_system_col}`
    """
    print(q)
    return spark.sql(q).toPandas()

source_counts_by_ss = get_counts_by_date_ts_source_system(
    spark, full_schema, SOURCE_TABLE, DATE_COL, TS_COL, SOURCE_SYSTEM_COL, ID_COLUMN
)
source_counts_by_ss = source_counts_by_ss.rename(columns={"distinct_count": "source_count"})

backup_counts_by_ss = get_counts_by_date_ts_source_system(
    spark, full_schema, BACKUP_TABLE, DATE_COL, TS_COL, SOURCE_SYSTEM_COL, ID_COLUMN
)
backup_counts_by_ss = backup_counts_by_ss.rename(columns={"distinct_count": "backup_count"})

merge_cols_ss = [DATE_COL, SOURCE_SYSTEM_COL]
combined_by_source_system = source_counts_by_ss.merge(
    backup_counts_by_ss,
    on=merge_cols_ss,
    how="outer",
    indicator=True
)
combined_by_source_system["source_count"] = combined_by_source_system["source_count"].fillna(0).astype(int)
combined_by_source_system["backup_count"] = combined_by_source_system["backup_count"].fillna(0).astype(int)
combined_by_source_system["diff"] = combined_by_source_system["source_count"] - combined_by_source_system["backup_count"]
combined_by_source_system["status"] = combined_by_source_system["diff"].apply(lambda x: "match" if x == 0 else "mismatch")
# combined_by_source_system["ingestion_timestamp_readable"] = combined_by_source_system[TS_COL].apply(ts_to_readable)
combined_by_source_system["in_source_only"] = combined_by_source_system["_merge"] == "left_only"
combined_by_source_system["in_backup_only"] = combined_by_source_system["_merge"] == "right_only"
combined_by_source_system = combined_by_source_system.drop(columns=["_merge"])

discrepancies_by_ss = combined_by_source_system[combined_by_source_system["status"] == "mismatch"]
print(f"Total (date, timestamp, source_system) groups: {len(combined_by_source_system)}")
print(f"Discrepancies (count mismatch or only in one side): {len(discrepancies_by_ss)}")
display(combined_by_source_system)
display(discrepancies_by_ss)

### 3c. Customers in mismatching rows (by source_system)

List **customersourceuniqueid** that are only in source or only in backup for each (ingestion_date, source_system) mismatch from 3b.

In [ ]:
def get_distinct_ids_by_source_system(spark, full_schema, table, date_col, source_system_col, id_col, date_val, source_system_val):
    # Escape string for SQL
    ss_val = str(source_system_val).replace("'", "''")
    q = f"""
    SELECT DISTINCT `{id_col}` AS id_val
    FROM {full_schema}.{table}
    WHERE `{date_col}` = '{date_val}' AND `{source_system_col}` = '{ss_val}' AND `{id_col}` IS NOT NULL
    """
    return set(r["id_val"] for r in spark.sql(q).collect())

MAX_IDS_LIST_SS = 500  # cap listed IDs per mismatch (None = no cap)

mismatch_customers_list = []
for _, row in discrepancies_by_ss.iterrows():
    d = row[DATE_COL]
    ss = row[SOURCE_SYSTEM_COL]
    src_ids = get_distinct_ids_by_source_system(
        spark, full_schema, SOURCE_TABLE, DATE_COL, SOURCE_SYSTEM_COL, ID_COLUMN, d, ss
    )
    bkp_ids = get_distinct_ids_by_source_system(
        spark, full_schema, BACKUP_TABLE, DATE_COL, SOURCE_SYSTEM_COL, ID_COLUMN, d, ss
    )
    only_in_source = sorted(src_ids - bkp_ids)
    only_in_backup = sorted(bkp_ids - src_ids)
    listed_src = only_in_source[:MAX_IDS_LIST_SS] if MAX_IDS_LIST_SS else only_in_source
    listed_bkp = only_in_backup[:MAX_IDS_LIST_SS] if MAX_IDS_LIST_SS else only_in_backup
    mismatch_customers_list.append({
        DATE_COL: str(d),
        SOURCE_SYSTEM_COL: ss,
        "source_count": int(row["source_count"]),
        "backup_count": int(row["backup_count"]),
        "only_in_source_count": len(only_in_source),
        "only_in_backup_count": len(only_in_backup),
        "only_in_source": listed_src,
        "only_in_backup": listed_bkp,
        "truncated": (MAX_IDS_LIST_SS is not None and (len(only_in_source) > MAX_IDS_LIST_SS or len(only_in_backup) > MAX_IDS_LIST_SS)),
    })

if len(mismatch_customers_list) == 0:
    print("No mismatching rows from 3b; no customers to list.")
else:
    for rec in mismatch_customers_list:
        cap_note = " (lists truncated)" if rec["truncated"] else ""
        print(f"--- {DATE_COL}={rec[DATE_COL]}, {SOURCE_SYSTEM_COL}={rec[SOURCE_SYSTEM_COL]} | "
              f"source_count={rec['source_count']}, backup_count={rec['backup_count']} | "
              f"only in source: {rec['only_in_source_count']}, only in backup: {rec['only_in_backup_count']}{cap_note} ---")
        if rec["only_in_source"] or rec["only_in_backup"]:
            df_src = pd.DataFrame({"customersourceuniqueid": rec["only_in_source"], "in_table": "source_only"}) if rec["only_in_source"] else pd.DataFrame()
            df_bkp = pd.DataFrame({"customersourceuniqueid": rec["only_in_backup"], "in_table": "backup_only"}) if rec["only_in_backup"] else pd.DataFrame()
            if len(df_src) > 0:
                print("Only in source:")
                display(df_src)
            if len(df_bkp) > 0:
                print("Only in backup:")
                display(df_bkp)

# Summary table (one row per mismatch with counts and list lengths)
mismatch_customers_df = pd.DataFrame([
    {DATE_COL: r[DATE_COL], SOURCE_SYSTEM_COL: r[SOURCE_SYSTEM_COL], "source_count": r["source_count"], "backup_count": r["backup_count"],
     "only_in_source_count": r["only_in_source_count"], "only_in_backup_count": r["only_in_backup_count"]}
    for r in mismatch_customers_list
])
print("Summary of mismatching rows and customer counts:")
display(mismatch_customers_df)

## 4. ID-level details for each discrepancy

In [ ]:
def get_distinct_ids(spark, full_schema, table, date_col, ts_col, id_col, date_val, ts_val):
    q = f"""
    SELECT DISTINCT `{id_col}` AS id_val
    FROM {full_schema}.{table}
    WHERE `{date_col}` = '{date_val}' AND `{ts_col}` = {int(ts_val)} AND `{id_col}` IS NOT NULL
    """
    return [r["id_val"] for r in spark.sql(q).collect()]

def collect_id_discrepancy(spark, full_schema, source_table, backup_table, id_col, date_col, ts_col, date_val, ts_val, max_ids):
    src_ids = set(get_distinct_ids(spark, full_schema, source_table, date_col, ts_col, id_col, date_val, ts_val))
    bkp_ids = set(get_distinct_ids(spark, full_schema, backup_table, date_col, ts_col, id_col, date_val, ts_val))
    only_in_source = list(src_ids - bkp_ids)
    only_in_backup = list(bkp_ids - src_ids)
    if max_ids is not None:
        only_in_source = only_in_source[:max_ids]
        only_in_backup = only_in_backup[:max_ids]
    return {
        "only_in_source": only_in_source,
        "only_in_source_total": len(src_ids - bkp_ids),
        "only_in_backup": only_in_backup,
        "only_in_backup_total": len(bkp_ids - src_ids),
        "truncated": max_ids is not None and (len(src_ids - bkp_ids) > max_ids or len(bkp_ids - src_ids) > max_ids),
    }

detail_list = []
for _, row in discrepancies_df.iterrows():
    date_val = row[DATE_COL]
    ts_val = row[TS_COL]
    detail = collect_id_discrepancy(
        spark, full_schema, SOURCE_TABLE, BACKUP_TABLE, ID_COLUMN,
        DATE_COL, TS_COL, date_val, ts_val, MAX_IDS_PER_DISCREPANCY
    )
    detail_list.append({
        DATE_COL: str(date_val),
        TS_COL: int(ts_val),
        "ingestion_timestamp_readable": ts_to_readable(ts_val),
        "source_count": int(row["source_count"]),
        "backup_count": int(row["backup_count"]),
        "diff": int(row["diff"]),
        **detail,
    })

print(f"Collected ID-level details for {len(detail_list)} discrepancy groups.")

## 5. Build and save JSON report

In [ ]:
report = {
    "metadata": {
        "generated_at": datetime.utcnow().isoformat() + "Z",
        "schema": SCHEMA,
        "catalog": CATALOG,
        "source_table": SOURCE_TABLE,
        "backup_table": BACKUP_TABLE,
        "id_column": ID_COLUMN,
        "date_column": DATE_COL,
        "timestamp_column": TS_COL,
    },
    "summary": {
        "total_groups": int(len(combined)),
        "match_count": int((combined["status"] == "match").sum()),
        "discrepancy_count": int(len(discrepancies_df)),
        "groups_only_in_source": int(combined["in_source_only"].sum()),
        "groups_only_in_backup": int(combined["in_backup_only"].sum()),
        "dates_with_mismatch": int(len(date_discrepancies)),
        "dates_latest_vs_previous_count_differ": len(latest_vs_previous_list),
    },
    "counts_by_date": by_date.to_dict(orient="records"),
    "counts_by_group": combined.drop(columns=["in_source_only", "in_backup_only"]).to_dict(orient="records"),
    "discrepancy_details": detail_list,
    "latest_vs_previous_timestamp_discrepancies": latest_vs_previous_list,
}

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, default=str)

print(f"Report saved to: {OUTPUT_JSON}")
print("Summary:", report["summary"])

## 6. Optional: Summary table and quick checks

In [ ]:
print("Discrepancies (ingestion_date, ingestion_timestamp, source_count, backup_count, diff):")
display(discrepancies_df[[DATE_COL, TS_COL, "ingestion_timestamp_readable", "source_count", "backup_count", "diff", "in_source_only", "in_backup_only"]])

In [ ]:
# Latest vs previous ingestion_timestamp discrepancies (per ingestion_date)
if len(latest_vs_previous_df) > 0:
    print("Dates where most recent and previous run have different distinct customersourceuniqueid:")
    display(latest_vs_previous_df)

In [ ]:
# Reload and inspect saved JSON (optional)
with open(OUTPUT_JSON, "r", encoding="utf-8") as f:
    loaded = json.load(f)
print("Keys:", list(loaded.keys()))
print("metadata:", loaded["metadata"])
print("summary:", loaded["summary"])